In [ ]:
!pip install transformers datasets torch librosa

In [ ]:
import torch
import numpy as np

from transformers import AutoFeatureExtractor, AutoModelForAudioClassification
from datasets import load_dataset, Audio, DatasetDict
from torch.utils.data import DataLoader

from scipy.special import softmax

import matplotlib.pyplot as plt

### Φόρτωση fine-tuned μοντέλου

In [ ]:
#Φόρτωση fine-tuned μοντέλου και feature extractor
repo_id = "VasilisAsim/whisper-finetuned-TESS"
model_ft = AutoModelForAudioClassification.from_pretrained(repo_id)
feature_extractor = AutoFeatureExtractor.from_pretrained(repo_id)

device = "cuda" if torch.cuda.is_available() else "cpu"
model_ft = model_ft.to(device)
model_ft.eval()

In [ ]:
label2id = model_ft.config.label2id
id2label = model_ft.config.id2label

### Φόρτωση dataset και προετοιμασία test split

In [ ]:
def load_tess_speaker_independent():
    """
    Συνάρτηση υπέυθηνη για τον speaker independent διαχωρισμό. Η ομιλητής OAF
    αντισοιχίζεται στο σύνολο εκπαίδευσης και η YAF στο σύνολο δοκιμής
    """
    dataset = load_dataset("AbstractTTS/TESS", split="train")
    dataset = dataset.cast_column("audio", Audio(decode=False))   # για το filter

    def get_speaker(example):
        filename = example["audio"]["path"]
        if "OAF" in filename:
            return "OAF"
        elif "YAF" in filename:
            return "YAF"
        return "unknown"

    train_dataset = dataset.filter(lambda x: get_speaker(x) == "OAF")
    test_dataset = dataset.filter(lambda x: get_speaker(x) == "YAF")

    train_dataset = train_dataset.cast_column("audio", Audio(sampling_rate=16_000))
    test_dataset = test_dataset.cast_column("audio", Audio(sampling_rate=16_000))

    return DatasetDict({"train": train_dataset, "test": test_dataset})

In [ ]:
dataset = load_tess_speaker_independent()

In [ ]:
#το Whisper feature extractor επιστρέφει "input_features" (log-mel spectrogram), όχι "input_values"
def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_features"] = feature_extractor(
        audio["array"], sampling_rate=audio["sampling_rate"]
    ).input_features[0]
    batch["labels"] = label2id[batch["emotion"]]
    return batch

encoded_dataset = dataset.map(prepare_dataset, remove_columns=dataset["train"].column_names, load_from_cache_file=False)

#Εφαρμογή στο speaker-independent test split (ομιλητής YAF)
encoded_test = dataset["test"].map(prepare_dataset, remove_columns=dataset["test"].column_names)

### Δημιουργία DataLoader

In [ ]:
#Padding των στοιχείων του dataset, ώστε να έχουν το ίδιο μήκος
def collate_fn(features):
    input_features = [{"input_features": f["input_features"]} for f in features]
    batch = feature_extractor.pad(input_features, padding=True, return_tensors="pt")
    batch["labels"] = torch.tensor(
        [f["labels"] for f in features], dtype=torch.long
    )
    return batch

test_loader = DataLoader(encoded_test, batch_size=8, collate_fn=collate_fn)


### Εξαγωγή logits στο test set

In [ ]:
#Πέρασμα του dataset μέσα απο το μοντέλο, συλλογή αποτελεσμάτων και label
all_logits, all_labels = [], []

with torch.no_grad():
    for batch in test_loader:
        input_values = batch["input_features"].to(device)
        labels = batch["labels"]

        #Πέρασμα κάθε στοιχείου από το finetuned μοντέλο και εξαγωγή των αποτελεσμάτων
        outputs = model_ft(input_features=input_values)

        #Αποθήκευση αποτελεσμάτων και label
        all_logits.append(outputs.logits.cpu().numpy())
        all_labels.append(labels.numpy())

#Αποθήκευση σε ενιαίο array
logits = np.concatenate(all_logits, axis=0)
labels = np.concatenate(all_labels, axis=0)

### Υπολογισμός μετρικών calibration & uncertainty

In [ ]:
probs = softmax(logits, axis=1)
predictions = np.argmax(probs, axis=1)
confidences = np.max(probs, axis=1)
correct = (predictions == labels)

num_classes = probs.shape[1]
labels_onehot = np.eye(num_classes)[labels]

def negative_log_likelihood(probs, labels, eps=1e-12):
    """
    Για κάθε δείγμα του συνόλου και πιθανότητα σωστής κλάσης το μοντέλο υπολογίζει το
    μέση -log(p_true) σε όλο το σύνολο δοκιμής.
    """
    p_true = np.clip(probs[np.arange(len(labels)), labels], eps, 1.0)
    return -np.mean(np.log(p_true))

nll = negative_log_likelihood(probs, labels)

def brier_score(probs, labels_onehot):
    """
    Υπολογίζει το μέσο τετραγωνικό σφάλμα ανάμεσα στο διάνυσμα πιθανοτήτων προβλέψεων και το
    διάνυσμα πραγματικής ετικέτας ανα δείγμα.
    """
    return np.mean(np.sum((probs - labels_onehot) ** 2, axis=1))

brier = brier_score(probs, labels_onehot)


In [ ]:
def expected_calibration_error(confidences, correct, n_bins=15):
    """
    Το εύρος [0, 1] χωρίζεται σε n binds βάσει του confidence κάθε πρόβλεψης και μετρά
    την απόλυτη διαφορά ανάμεσα σε accuracy και mean confidence εντός κάθε bin.
    """
    bin_edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    bin_info = []
    for i in range(n_bins):
        lo, hi = bin_edges[i], bin_edges[i + 1]
        in_bin = (confidences > lo) & (confidences <= hi) if i > 0 else (confidences >= lo) & (confidences <= hi)
        prop = np.mean(in_bin)
        if prop > 0:
            acc_bin = np.mean(correct[in_bin])
            conf_bin = np.mean(confidences[in_bin])
            ece += prop * abs(acc_bin - conf_bin)
            bin_info.append((lo, hi, acc_bin, conf_bin, prop))
    return ece, bin_info

ece, ece_bins = expected_calibration_error(confidences, correct, n_bins=15)

In [ ]:
def adaptive_calibration_error(confidences, correct, n_bins=15):
    """
    Τα δείγματα ταξινομούνται βάση confidence σε bin με ίσο αριθμό δειγμάτων το καθένα
    και η ίδια διαφορά υπολογίζεται εκ νέου.
    """
    order = np.argsort(confidences)
    conf_sorted = confidences[order]
    correct_sorted = correct[order]
    splits = np.array_split(np.arange(len(conf_sorted)), n_bins)

    ace = 0.0
    bin_info = []
    for idx in splits:
        if len(idx) == 0:
            continue
        acc_bin = np.mean(correct_sorted[idx])
        conf_bin = np.mean(conf_sorted[idx])
        prop = len(idx) / len(conf_sorted)
        ace += prop * abs(acc_bin - conf_bin)
        bin_info.append((conf_bin, acc_bin, prop))
    return ace, bin_info

ace, ace_bins = adaptive_calibration_error(confidences, correct, n_bins=15)

### Οπτικοποίηση αποτελεσμάτων

In [ ]:
#Reliability diagram (Βάσει του ECE)
def plot_reliability_diagram(bin_info, title="Reliability Diagram"):
    confs = [b[3] for b in bin_info]
    accs = [b[2] for b in bin_info]
    plt.figure(figsize=(5, 5))
    plt.bar(confs, accs, width=0.05, edgecolor="black", alpha=0.7, label="Model")
    plt.plot([0, 1], [0, 1], "k--", label="Perfect calibration")
    plt.xlabel("Confidence")
    plt.ylabel("Accuracy")
    plt.title(title)
    plt.legend()
    plt.xlim(0, 1)
    plt.ylim(0, 1)
    plt.show()

plot_reliability_diagram(ece_bins, title=f"Reliability Diagram (ECE={ece:.4f})")


#Predictive Entropy
def predictive_entropy(probs, eps=1e-12):
    return -np.sum(probs * np.log(probs + eps), axis=1)

entropy = predictive_entropy(probs)

plt.figure(figsize=(6, 4))
plt.hist(entropy[correct], bins=30, alpha=0.6, label="Σωστές προβλέψεις")
plt.hist(entropy[~correct], bins=30, alpha=0.6, label="Λάθος προβλέψεις")
plt.xlabel("Predictive Entropy")
plt.ylabel("Πλήθος δειγμάτων")
plt.title("Κατανομή Predictive Entropy")
plt.legend()
plt.show()


#Confidence distribution: σωστές vs λάθος
plt.figure(figsize=(6, 4))
plt.hist(confidences[correct], bins=30, alpha=0.6, label="Σωστές προβλέψεις")
plt.hist(confidences[~correct], bins=30, alpha=0.6, label="Λάθος προβλέψεις")
plt.xlabel("Confidence (max probability)")
plt.ylabel("Πλήθος δειγμάτων")
plt.title("Κατανομή Confidence: Σωστά vs Λάθος")
plt.legend()
plt.show()


#Σύνοψη
print(f"NLL: {nll:.4f}")
print(f"Brier Score: {brier:.4f}")
print(f"ECE: {ece:.4f}")
print(f"ACE: {ace:.4f}")